# 07 — Multi-Agent Workflow: Bounded Scientific Coordination

**CuSelect-AI · campaign CU-001 · original ten-notebook roadmap**

**Scientific question:** Can a Literature Agent, a constrained Structure Design Agent, and a preliminary Scientific Critic role coordinate **reproducibly**, without inventing observations, claiming unvalidated catalyst selectivity, or changing human-review decisions?

**Milestone scope:** An auditable **deterministic multi-role prototype**, not autonomous LLM agents. Full falsification/critic development is reserved for **Notebook 08**; independent benchmarking is reserved for **Notebook 09**.

**Workflow:** request → bounded routing → approved-evidence lookup / existing geometry selection → policy-level scientific review → structured trace → optional isolated export.

- **Literature:** reads only the three synchronized, human-verified experimental claims exported from Notebook 06.
- **Design:** proposes *existing* Cu/Ag geometries from Notebook 04/05; does not synthesize or score a new catalyst.
- **Scientific Critic (minimal gate):** distinguishes published observations, geometry hypotheses, and unsupported predictions. It can refuse claims but cannot create modeling approval.
- **Coordinator:** enforces a step budget, typed task schema, deterministic ordering, traceable tool invocation, and no network or files written by default.

**Foundational boundary:** One evidence-bearing publication and eight unrelaxed, not-yet-approved geometries do not support intrinsic ethylene/ethanol rankings, computed adsorption energies, or electrochemical selectivity predictions.

## 7.1 — Reproducibility contract and authoritative inputs

**Why:** The agents must start from the saved handoff artifacts, *not* a notebook kernel's hidden state. Require the specific versioned-source relationships and keep production-readiness flags closed. A missing export must fail with a precise explanation; it must never be silently regenerated.

In [2]:
# PSEUDOCODE
# 1. Find the repository root and exact input files produced by Notebooks 02, 04, 05, and 06.
# 2. Refuse to proceed if any needed export is missing.
# 3. Load CSV and JSON inputs without changing strings or assigning approvals.
# 4. Establish bounded, offline, read-only defaults.
# 5. Describe inputs and restrictions.

from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Literal
from IPython.display import display
from collections import Counter
import hashlib
import json
import re
import pandas as pd

_here07 = Path.cwd().resolve()
ROOT = next((p for p in (_here07, *_here07.parents) if (p / 'pyproject.toml').is_file()), None)
assert ROOT is not None, 'Open the notebook inside the CuSelect-AI project.'

paths07 = {
    'papers': ROOT / 'data/papers.csv',
    'human_review': ROOT / 'data/references/02_human_reviewed_claims.csv',
    'verified_claims': ROOT / 'data/processed/02_verified_claims.csv',
    'readiness02': ROOT / 'outputs/reports/02_evidence_readiness.json',
    'structures04': ROOT / 'data/processed/04_structures/04_structure_registry.csv',
    'structures_manifest04': ROOT / 'data/processed/04_structures/04_structure_manifest.json',
    'features05': ROOT / 'data/processed/05_geometry_descriptors/05_geometry_descriptors.csv',
    'features_manifest05': ROOT / 'data/processed/05_geometry_descriptors/05_descriptor_manifest.json',
    'corpus06': ROOT / 'data/processed/06_rag_prototype/06_retrieval_corpus.jsonl',
    'corpus_manifest06': ROOT / 'data/processed/06_rag_prototype/06_retrieval_manifest.json',
}
missing07 = [str(p) for p in paths07.values() if not p.is_file()]
assert not missing07, 'Missing prior notebook exports. Finish their manual export gates first:\n' + '\n'.join(missing07)

papers07 = pd.read_csv(paths07['papers'], dtype=str, keep_default_na=False)
reviews07 = pd.read_csv(paths07['human_review'], dtype=str, keep_default_na=False)
claims07 = pd.read_csv(paths07['verified_claims'], dtype=str, keep_default_na=False)
registry07 = pd.read_csv(paths07['structures04'], dtype=str, keep_default_na=False)
features07 = pd.read_csv(paths07['features05'], dtype=str, keep_default_na=False)
readiness07 = json.loads(paths07['readiness02'].read_text(encoding='utf-8'))
structure_manifest07 = json.loads(paths07['structures_manifest04'].read_text(encoding='utf-8'))
feature_manifest07 = json.loads(paths07['features_manifest05'].read_text(encoding='utf-8'))
corpus_manifest07 = json.loads(paths07['corpus_manifest06'].read_text(encoding='utf-8'))
corpus_bytes07 = paths07['corpus06'].read_bytes()
corpus07 = [json.loads(line) for line in corpus_bytes07.decode('utf-8').splitlines() if line.strip()]

MAX_AGENT_STEPS07 = 3
MAX_CANDIDATES07 = 2
MAX_EVIDENCE07 = 3
ALLOW_EXTERNAL_LLM07 = False
RUN_COMPUTATIONAL_SCORING07 = False
WRITE_AGENT_TRACE07 = False

assert readiness07['campaign_id'] == 'CU-001'
assert readiness07['evidence_ready_for_rag'] is False
assert readiness07['ready_for_blinded_benchmark'] is False
print('NOTEBOOK 07 — REPRODUCIBILITY CONTRACT')
print('Project root:', ROOT)
print('Approved experimental claims:', len(claims07))
print('Exported retrieval documents:', len(corpus07))
print('Exported atomic models:', len(registry07))
print('Geometry descriptor rows:', len(features07))
print('Network / computational scoring / export: DISABLED')
print('No source files modified')

NOTEBOOK 07 — REPRODUCIBILITY CONTRACT
Project root: /Users/nsolebna/202609_Ironhack/Week 8/Project 3/CuSelect-AI
Approved experimental claims: 3
Exported retrieval documents: 3
Exported atomic models: 8
Geometry descriptor rows: 8
Network / computational scoring / export: DISABLED
No source files modified


## 7.2 — Cross-notebook admission and provenance gates

**Why:** A successful export does not grant modeling approval. Verify byte-level corpus integrity, exact claim synchronization with the human review register, DOI relationships, geometry identities, and explicit unrelaxed/unapproved labels. If anything differs, *stop* rather than repairing records.

In [3]:
# PSEUDOCODE
# 1. Verify the Notebook 06 corpus hash and its three approved IDs.
# 2. Compare the full experimental review records against their approved export.
# 3. Verify paper/DOI linkage and source location for each retrieval document.
# 4. Compare the Notebook 04 registry to Notebook 05 descriptors by ID and hash.
# 5. Confirm all geometry-related modeling and readiness gates remain closed.

assert hashlib.sha256(corpus_bytes07).hexdigest() == corpus_manifest07['corpus_sha256'], 'RAG corpus hash mismatch'
assert len(papers07) == 8 and papers07['paper_id'].is_unique
assert claims07['evidence_id'].is_unique and reviews07['evidence_id'].is_unique
approved07 = reviews07.loc[
    reviews07['review_status'].eq('human_verified')
    & reviews07['source_checked_by_human'].str.lower().str.strip().eq('true')
]
cols07 = sorted(claims07.columns)
assert set(cols07) == set(reviews07.columns)
pd.testing.assert_frame_equal(
    claims07[cols07].sort_values('evidence_id').reset_index(drop=True),
    approved07[cols07].sort_values('evidence_id').reset_index(drop=True),
    check_dtype=False,
)
assert len(claims07) == len(corpus07) == 3
by_id07 = {r['evidence_id']:r for r in claims07.to_dict('records')}
corpus_by_id07 = {r['evidence_id']:r for r in corpus07}
assert len(corpus_by_id07) == 3
paper_doi07 = dict(zip(papers07['paper_id'], papers07['doi'].str.lower().str.strip()))
for item in corpus07:
    src = by_id07[item['evidence_id']]
    assert item['review_status'] == src['review_status'] == 'human_verified'
    for field in ['paper_id','doi','claim_paraphrase','conditions','source_extent',
                  'catalyst_system','target_or_observation','fe_product','faradaic_efficiency_percent']:
        assert str(item[field]) == str(src[field]), (item['evidence_id'],field)
    assert item['source_location'] == src['exact_source_location']
    assert paper_doi07[item['paper_id']] == item['doi'].lower().strip()

assert registry07['candidate_id'].is_unique and features07['candidate_id'].is_unique
assert len(registry07) == len(features07) == 8
assert set(registry07['candidate_id']) == set(features07['candidate_id'])
assert registry07['geometry_sha256'].is_unique
assert structure_manifest07['candidate_count'] == feature_manifest07['candidate_count'] == 8
assert structure_manifest07['ml_approved'] is False
assert feature_manifest07['modeling_approved'] is False
assert not registry07['approved_for_modeling'].str.lower().eq('true').any()
assert not features07['approved_for_modeling'].str.lower().eq('true').any()
assert registry07['geometry_status'].eq('unrelaxed').all()
assert registry07['structure_review_status'].eq('pending_visual_review').all()
assert features07['descriptor_scope'].eq('geometry_only_unrelaxed').all()
assert feature_manifest07['DFT_reference_labels_included'] is False
assert feature_manifest07['experimental_selectivity_targets_included'] is False
fingerprint07 = dict(zip(registry07['candidate_id'], registry07['geometry_sha256']))
for f in features07.to_dict('records'):
    assert f['geometry_sha256'] == fingerprint07[f['candidate_id']]
    assert feature_manifest07['fingerprints'][f['candidate_id']] == f['geometry_sha256']
for cid in registry07['candidate_id']:
    assert (paths07['structures04'].parent / f'{cid}.extxyz').is_file(), ('Missing atomic model',cid)

assert len({x['paper_id'] for x in corpus07}) == 1
print('CROSS-NOTEBOOK ADMISSION: PASSED')
print('Human-verified, synchronized claims:', len(by_id07))
print('Verified geometry identities and fingerprints:', len(fingerprint07))
print('Independent evidence-bearing papers:', len(set(x['paper_id'] for x in corpus07)))
print('ML / selectivity / production RAG approval: NOT ESTABLISHED')
print('No source files modified')

CROSS-NOTEBOOK ADMISSION: PASSED
Human-verified, synchronized claims: 3
Verified geometry identities and fingerprints: 8
Independent evidence-bearing papers: 1
ML / selectivity / production RAG approval: NOT ESTABLISHED
No source files modified


## 7.3 — Define typed tasks, bounded actions, and explicit agent roles

**Why:** A multi-agent interface without a tool schema, bounded execution, or reproducible trace is not yet a testable research workflow. The roles below are *functions with limited permissions*; there is no model-powered autonomy.

**Role permissions:** Literature → approved claim retrieval; Design → registered geometry selection; Preliminary Critic → policy and evidence checks; Coordinator → bounded orchestration and logging. The Critic cannot approve candidate geometries or scientific claims.

In [4]:
# PSEUDOCODE
# 1. Define immutable request, retrieval, proposal, and review objects.
# 2. Enumerate permitted request types; reject unknown types.
# 3. Define small, explicit tool allowlists by agent role.
# 4. Impose maximum steps, records, and candidate limits before execution.
# 5. Keep human review, scoring, API usage, and file writes unavailable to roles.

TaskType07 = Literal['source_question','structure_hypothesis','predict_selectivity']
Decision07 = Literal['evidence_only','hypothesis_only','abstain']

@dataclass(frozen=True)
class ResearchTask07:
    task_id: str
    task_type: TaskType07
    question: str
    candidate_ids: tuple[str, ...] = ()

@dataclass(frozen=True)
class EvidenceResult07:
    ids: tuple[str, ...]
    independent_publications: int
    status: str

@dataclass(frozen=True)
class DesignResult07:
    ids: tuple[str, ...]
    status: str

@dataclass(frozen=True)
class ReviewResult07:
    decision: Decision07
    reasons: tuple[str, ...]
    citation_ids: tuple[str, ...]
    candidate_ids: tuple[str, ...]
    modeling_approved: bool = False

AGENT_TOOLS07 = {
    'literature_agent':frozenset({'approved_claim_lookup'}),
    'design_agent':frozenset({'registered_geometry_lookup'}),
    'scientific_critic':frozenset({'policy_check'}),
}
assert MAX_AGENT_STEPS07 == 3 and MAX_EVIDENCE07 == 3 and MAX_CANDIDATES07 == 2
assert all('external_api' not in allowed and 'compute_energy' not in allowed
           for allowed in AGENT_TOOLS07.values())
print('TYPED AGENT CONTRACT: DEFINED')
print('Agent tools:',{name:sorted(tools) for name,tools in AGENT_TOOLS07.items()})
print('Max steps / evidence / candidates:',MAX_AGENT_STEPS07,MAX_EVIDENCE07,MAX_CANDIDATES07)

TYPED AGENT CONTRACT: DEFINED
Agent tools: {'literature_agent': ['approved_claim_lookup'], 'design_agent': ['registered_geometry_lookup'], 'scientific_critic': ['policy_check']}
Max steps / evidence / candidates: 3 3 2


## 7.4 — Literature Agent: narrow, evidence-grounded lookup

**Why:** Notebook 06 demonstrated lexical retrieval, but a ranking score is not scientific support. This agent selects records only when a *narrowly recognized query* matches an approved experimental measurement. All broader questions abstain by default. This is intentionally a conservative prototype, not a general-language question parser.

In [5]:
# PSEUDOCODE
# 1. Normalize typography and recognize only a small whitelist of measurement questions.
# 2. Reject durability, DFT, predicted-performance, ranking and mechanism requests.
# 3. Match an FE product and catalyst identity, or the verified product-ratio observation.
# 4. Return at most three approved evidence IDs, no inferred claims.
# 5. Always preserve DOI and source provenance through immutable claim records.

UNSUPPORTED_TERMS07 = {'durability','stability','100 hour','100-hour','best catalyst',
                       'optimal','predict','prediction','forecast','dft','adsorption',
                       'mechanism','activation barrier','selectivity ranking'}

def literature_agent07(task:ResearchTask07) -> EvidenceResult07:
    q = task.question.lower().replace('₁','1').replace('₂','2').replace('–','-').strip()
    if task.task_type != 'source_question':
        return EvidenceResult07((),0,'not_applicable')
    if any(term in q for term in UNSUPPORTED_TERMS07):
        return EvidenceResult07((),0,'unsupported_question')
    selected = []
    if ('ratio' in q or 'ratios' in q) and 'ethanol' in q and 'ethylene' in q:
        selected = [r for r in corpus07 if r['target_or_observation'].lower().find('ratio') >= 0]
    elif ('fe' in q or 'faradaic' in q or 'efficiency' in q) and 'ethylene' in q and 'agcu nw' in q and 'ag1cu nw' not in q:
        selected = [r for r in corpus07 if r['fe_product'].upper()=='C2H4' and r['catalyst_system'].strip()=='AgCu NW']
    elif ('fe' in q or 'faradaic' in q or 'efficiency' in q) and 'ethanol' in q and 'ag1cu nw' in q:
        selected = [r for r in corpus07 if r['fe_product'].upper()=='C2H5OH' and r['catalyst_system'].strip()=='Ag1Cu NW']
    selected = selected[:MAX_EVIDENCE07]
    assert all(r['evidence_id'] in by_id07 for r in selected)
    return EvidenceResult07(
        tuple(r['evidence_id'] for r in selected),
        len({r['paper_id'] for r in selected}),
        'matched_reviewed_observation' if selected else 'insufficient_retrieval_support',
    )

examples_lit07 = [
    ResearchTask07('L-01','source_question','What ethylene FE was reported for AgCu NW?'),
    ResearchTask07('L-02','source_question','What ethanol FE was reported for Ag1Cu NW?'),
    ResearchTask07('L-03','source_question','What ethanol-to-ethylene product ratios were reported?'),
    ResearchTask07('L-04','source_question','What was the 100-hour durability of AgCu NW?'),
]
for t in examples_lit07:
    print(t.task_id, '->', literature_agent07(t))
assert literature_agent07(examples_lit07[0]).ids == ('E-P005-FE-C2H4-001',)
assert literature_agent07(examples_lit07[1]).ids == ('E-P005-FE-ETOH-001',)
assert literature_agent07(examples_lit07[2]).ids == ('E-DRAFT-001',)
assert literature_agent07(examples_lit07[3]).ids == ()
print('LITERATURE AGENT: conservative admission checks PASSED')

L-01 -> EvidenceResult07(ids=('E-P005-FE-C2H4-001',), independent_publications=1, status='matched_reviewed_observation')
L-02 -> EvidenceResult07(ids=('E-P005-FE-ETOH-001',), independent_publications=1, status='matched_reviewed_observation')
L-03 -> EvidenceResult07(ids=('E-DRAFT-001',), independent_publications=1, status='matched_reviewed_observation')
L-04 -> EvidenceResult07(ids=(), independent_publications=0, status='unsupported_question')
LITERATURE AGENT: conservative admission checks PASSED


## 7.5 — Structure Design Agent: controlled proposals, not predictions

**Why:** The Design Agent must not mistake a Cu(100) Ag-pair motif for a demonstrated catalyst improvement. It can propose **existing registered geometries**, while preserving their unrelaxed and pending-visual-review status. New atomistic generation and physical screening remain future work.

In [6]:
# PSEUDOCODE
# 1. Accept a task only when it requests a structural hypothesis.
# 2. Select at most two candidate IDs already present in the verified geometry registry.
# 3. Require matched Cu(100) composition for the designed pair-comparison example.
# 4. Return candidate identities without energies, performance ranks or approval.
# 5. Reject missing or unknown candidate IDs rather than inventing a geometry.

registered07 = registry07.set_index('candidate_id',drop=False)
feature_by_id07 = features07.set_index('candidate_id',drop=False)
PAIRED_CU10007 = ('CU-001-100-AG2-NN','CU-001-100-AG2-SEP')

def design_agent07(task:ResearchTask07) -> DesignResult07:
    if task.task_type != 'structure_hypothesis':
        return DesignResult07((),'not_applicable')
    ids = task.candidate_ids
    if not ids or len(ids)>MAX_CANDIDATES07 or len(ids)!=len(set(ids)):
        return DesignResult07((),'invalid_candidate_request')
    if any(cid not in registered07.index for cid in ids):
        return DesignResult07((),'unknown_candidate_id')
    if ids == PAIRED_CU10007:
        a,b = (registered07.loc[cid] for cid in ids)
        assert a['chemical_formula'] == b['chemical_formula'] == 'Ag2Cu34'
        assert a['facet'] == b['facet'] == '100'
    for cid in ids:
        item=registered07.loc[cid]
        assert item['geometry_status']=='unrelaxed'
        assert item['structure_review_status']=='pending_visual_review'
        assert item['approved_for_modeling'].strip().lower()=='false'
    return DesignResult07(tuple(ids),'existing_unrelaxed_hypotheses_only')

pair_task07=ResearchTask07('D-01','structure_hypothesis',
                           'Compare local geometry of adjacent versus separated Ag pairs on Cu(100)',
                           PAIRED_CU10007)
print('DESIGN AGENT EXAMPLE:',design_agent07(pair_task07))
assert design_agent07(pair_task07).ids == PAIRED_CU10007
assert design_agent07(ResearchTask07('D-02','structure_hypothesis','Unknown geometry',('FAKE-CANDIDATE',))).ids == ()
print('No candidate generated, relaxed, ranked, or approved')

DESIGN AGENT EXAMPLE: DesignResult07(ids=('CU-001-100-AG2-NN', 'CU-001-100-AG2-SEP'), status='existing_unrelaxed_hypotheses_only')
No candidate generated, relaxed, ranked, or approved


## 7.6 — Preliminary Scientific Critic: policy-level review only

**Why:** We need a minimum gate before agents communicate, but **Notebook 08** is where mechanistic criticism, falsifiable objections, abstention/false-advancement evaluation, and independent reviewer checks will be implemented. Here, the gate only enforces clearly established boundaries.

In [7]:
# PSEUDOCODE
# 1. Check that any cited evidence IDs belong to the authorized corpus.
# 2. Check that every proposed geometry exists and remains unapproved for modeling.
# 3. Allow a narrow extractive observation, but block claims of intrinsic superiority.
# 4. Treat candidate selections as hypotheses needing visual/energetic validation.
# 5. Abstain on unsupported selectivity, missing evidence, and invalid candidate IDs.


def preliminary_critic07(task:ResearchTask07,
                          evidence:EvidenceResult07,
                          proposal:DesignResult07) -> ReviewResult07:
    assert set(evidence.ids) <= set(corpus_by_id07), 'Unapproved/fabricated citation rejected'
    assert set(proposal.ids) <= set(registered07.index), 'Unknown geometry rejected'
    if task.task_type=='predict_selectivity':
        return ReviewResult07('abstain',(
            'No validated catalyst-selectivity model or matched experimental targets.',
            'Geometry and generic atomistic energies cannot establish CO2RR Faradaic efficiency.',
        ),(),())
    if task.task_type=='source_question':
        if not evidence.ids:
            return ReviewResult07('abstain',('No matching human-verified observation for this question.',),(),())
        return ReviewResult07('evidence_only',(
            'Citation-linked, published observation only; one source publication.',
            'No inference about catalyst ranking, reaction mechanism or durability.',
        ),evidence.ids,())
    if task.task_type=='structure_hypothesis':
        if not proposal.ids:
            return ReviewResult07('abstain',('Candidate ID missing, unknown, or invalid.',),(),())
        return ReviewResult07('hypothesis_only',(
            'Unrelaxed geometries: pending formal visual review.',
            'No reference DFT, energy ranking, or measured selectivity for these geometries.',
        ),(),proposal.ids)
    raise ValueError('Unknown task type')

print('PRELIMINARY CRITIC POLICY: DEFINED')
print('Review cannot change approved evidence or geometry flags')

PRELIMINARY CRITIC POLICY: DEFINED
Review cannot change approved evidence or geometry flags


## 7.7 — Coordinator: bounded, deterministic, typed orchestration

**Why:** We need an explicit execution state and tool invocation ledger. Each role receives only authorized inputs. The run is finite and reproducible: **Literature → Design → Critic**, maximum three calls. Empty/irrelevant role outputs are reported honestly; they are not hidden. There is no asynchronous agent or model call.

In [8]:
# PSEUDOCODE
# 1. Accept only validated task types and unique, bounded candidate IDs.
# 2. Invoke exactly three role-specific, allowlisted tools in fixed order.
# 3. Record the action, input/output fingerprints, and a concise role message for each call.
# 4. Run the critic on both independent role outputs and preserve its final decision.
# 5. Return an immutable trace plus a deterministic run digest; do not write files.


def canonical07(value):
    return json.dumps(value,sort_keys=True,separators=(',',':'),ensure_ascii=False,default=str)

def digest07(value):
    return hashlib.sha256(canonical07(value).encode('utf-8')).hexdigest()


def run_workflow07(task:ResearchTask07):
    if task.task_type not in ('source_question','structure_hypothesis','predict_selectivity'):
        raise ValueError('Unrecognized task type')
    if not task.task_id.strip() or not task.question.strip():
        raise ValueError('task_id and question must not be empty')
    if len(task.candidate_ids)>MAX_CANDIDATES07:
        raise ValueError('Candidate request exceeds the maximum allowed size')
    trace=[]
    def log(agent,tool,inputs,output,brief):
        assert tool in AGENT_TOOLS07[agent], 'Tool not allowed for role'
        assert len(trace)<MAX_AGENT_STEPS07, 'Step budget exhausted'
        trace.append({
            'step':len(trace)+1,'agent':agent,'tool':tool,
            'input_sha256':digest07(inputs),'output_sha256':digest07(asdict(output)),
            'message':brief,
        })
    e=literature_agent07(task)
    log('literature_agent','approved_claim_lookup',asdict(task),e,
        f"Approved claim IDs: {list(e.ids)}; status: {e.status}")
    d=design_agent07(task)
    log('design_agent','registered_geometry_lookup',asdict(task),d,
        f"Registered candidate IDs: {list(d.ids)}; status: {d.status}")
    c=preliminary_critic07(task,e,d)
    log('scientific_critic','policy_check',
        {'task':asdict(task),'evidence':asdict(e),'proposal':asdict(d)},c,
        f"Decision: {c.decision}; reasons: {' | '.join(c.reasons)}")
    assert len(trace)<=MAX_AGENT_STEPS07
    result = {
        'task':asdict(task),
        'literature':asdict(e),
        'design':asdict(d),
        'critic':asdict(c),
        'trace':trace,
        'source_corpus_sha256':corpus_manifest07['corpus_sha256'],
        'workflow_type':'deterministic_local_no_llm',
        'external_api_called':False,
        'predicted_selectivity':False,
        'human_approval_modified':False,
    }
    result['run_sha256']=digest07(result)
    return result

print('COORDINATOR: DEFINED')
print('Execution is bounded to',MAX_AGENT_STEPS07,'tool calls per request')

COORDINATOR: DEFINED
Execution is bounded to 3 tool calls per request


## 7.8 — Run three scientific cases and visualize the agent dialogue

**Case A:** retrieve a reported experimental value; **Case B:** propose a controlled matched-composition structural comparison; **Case C:** request an unsupported selectivity ranking. The intended decisions differ: `evidence_only`, `hypothesis_only`, and `abstain`.

This section reports role messages and candidate status transitions, **not** calculated energies, learned policies, or a simulated scientific debate.

In [9]:
# PSEUDOCODE
# 1. Define three immutable, independently interpretable science tasks.
# 2. Run the same bounded coordinator on each task.
# 3. Display decision, evidence IDs, candidate IDs, and any refusal reasons.
# 4. Display chronological tool messages as an auditable 'agent dialogue'.
# 5. Do not generate scientific performance values.

workflow_tasks07 = [
    ResearchTask07('W07-01','source_question','What ethylene Faradaic efficiency was reported for AgCu NW?'),
    ResearchTask07('W07-02','structure_hypothesis',
                   'Compare neighboring and separated Ag pairs on Cu(100) as geometric hypotheses',PAIRED_CU10007),
    ResearchTask07('W07-03','predict_selectivity','Which Ag-Cu catalyst will have the highest ethylene FE?'),
]
workflow_runs07=[run_workflow07(t) for t in workflow_tasks07]
assert [r['critic']['decision'] for r in workflow_runs07] == [
    'evidence_only','hypothesis_only','abstain']

decision_rows07=[{
    'task_id':r['task']['task_id'],
    'request_type':r['task']['task_type'],
    'decision':r['critic']['decision'],
    'cited_ids':', '.join(r['critic']['citation_ids']) or '—',
    'candidates':', '.join(r['critic']['candidate_ids']) or '—',
    'steps':len(r['trace']),
    'run_sha256_prefix':r['run_sha256'][:12],
} for r in workflow_runs07]
trace_rows07=[{'task_id':r['task']['task_id'],**event} for r in workflow_runs07 for event in r['trace']]
decisions_df07=pd.DataFrame(decision_rows07)
trace_df07=pd.DataFrame(trace_rows07)
print('MULTI-AGENT WORKFLOW DECISIONS')
display(decisions_df07)
print('BOUNDED AGENT DIALOGUE / TOOL TRACE')
display(trace_df07[['task_id','step','agent','tool','message']])
print('No model execution, human approval, or file write occurred')

MULTI-AGENT WORKFLOW DECISIONS


,task_id,request_type,decision,cited_ids,candidates,steps,run_sha256_prefix
0,W07-01,source_question,evidence_only,E-P005-FE-C2H4-001,—,3,bb2216026490
1,W07-02,structure_hypothesis,hypothesis_only,—,"CU-001-100-AG2-NN, CU-001-100-AG2-SEP",3,fa3e11a21627
2,W07-03,predict_selectivity,abstain,—,—,3,7d6ba8d1b7b9


BOUNDED AGENT DIALOGUE / TOOL TRACE


,task_id,step,agent,tool,message
0,W07-01,1,literature_agent,approved_claim_lookup,Approved claim IDs: ['E-P005-FE-C2H4-001']; st...
1,W07-01,2,design_agent,registered_geometry_lookup,Registered candidate IDs: []; status: not_appl...
2,W07-01,3,scientific_critic,policy_check,Decision: evidence_only; reasons: Citation-lin...
3,W07-02,1,literature_agent,approved_claim_lookup,Approved claim IDs: []; status: not_applicable
4,W07-02,2,design_agent,registered_geometry_lookup,Registered candidate IDs: ['CU-001-100-AG2-NN'...
5,W07-02,3,scientific_critic,policy_check,Decision: hypothesis_only; reasons: Unrelaxed ...
6,W07-03,1,literature_agent,approved_claim_lookup,Approved claim IDs: []; status: not_applicable
7,W07-03,2,design_agent,registered_geometry_lookup,Registered candidate IDs: []; status: not_appl...
8,W07-03,3,scientific_critic,policy_check,Decision: abstain; reasons: No validated catal...


No model execution, human approval, or file write occurred


## 7.9 — Candidate progression: geometry proposal → review-required queue

**Why:** The original flagship concept includes visualization of candidate evolution. At this stage, **the only defensible transition** is from a *registered unrelaxed geometry* to a *proposed geometry requiring independent evaluation*. No candidate has advanced through a physical-performance gate.

In [10]:
# PSEUDOCODE
# 1. Select only the two requested Cu(100) Ag2 motifs from the registered geometry library.
# 2. Display original geometry metadata and comparison-only structural descriptors.
# 3. Create a deterministic human-review queue with no automatic approvals.
# 4. Display a simple progression table; never rank candidates by catalytic performance.

pair_run07=next(r for r in workflow_runs07 if r['task']['task_id']=='W07-02')
assert pair_run07['critic']['decision']=='hypothesis_only'
progress_rows07=[]
for cid in pair_run07['critic']['candidate_ids']:
    base=registered07.loc[cid]
    feat=feature_by_id07.loc[cid]
    progress_rows07.append({
        'candidate_id':cid,'facet':base['facet'],
        'formula':base['chemical_formula'],'motif':base['motif'],
        'ag_ag_distance_A':feat['ag_ag_min_image_A'],
        'cu_ag_near_pair_count':int(feat['cu_ag_near_pair_count']),
        'initial_state':'unrelaxed_registered_geometry',
        'proposed_state':'hypothesis_for_review',
        'next_gate':'human_visual_review_and_independent_reference_calculation',
        'energy_eV':'NOT_CALCULATED',
        'predicted_FE_percent':'NOT_AVAILABLE',
        'approved_for_modeling':False,
    })
progress_df07=pd.DataFrame(progress_rows07)
assert len(progress_df07)==2
assert not progress_df07['approved_for_modeling'].any()
print('CANDIDATE PROGRESSION — NO SCIENTIFIC ADVANCEMENT CLAIMED')
display(progress_df07)
print('Visual sequence: registered geometry → design hypothesis → pending review')

CANDIDATE PROGRESSION — NO SCIENTIFIC ADVANCEMENT CLAIMED


,candidate_id,facet,formula,motif,ag_ag_distance_A,cu_ag_near_pair_count,initial_state,proposed_state,next_gate,energy_eV,predicted_FE_percent,approved_for_modeling
0,CU-001-100-AG2-NN,100,Ag2Cu34,top_layer_nearest_pair,2.5561910099999996,14,unrelaxed_registered_geometry,hypothesis_for_review,human_visual_review_and_independent_reference_...,NOT_CALCULATED,NOT_AVAILABLE,False
1,CU-001-100-AG2-SEP,100,Ag2Cu34,top_layer_separated_pair,3.6149999943581794,16,unrelaxed_registered_geometry,hypothesis_for_review,human_visual_review_and_independent_reference_...,NOT_CALCULATED,NOT_AVAILABLE,False


Visual sequence: registered geometry → design hypothesis → pending review


## 7.10 — Extractive evidence response and citation checking

**Why:** An agent trace is useful only if the final answer is separately auditable. The response below is copied from the approved claim's *reviewed paraphrase*, with explicit source location, DOI and operating conditions. It is not LLM-generated or independent evidence.

In [11]:
# PSEUDOCODE
# 1. Take a run whose preliminary critic permits source-only reporting.
# 2. Construct an extractive statement from the exact approved claim.
# 3. Attach DOI, source location, conditions and evidence ID.
# 4. Verify every displayed scientific claim points to a retrieved approved record.
# 5. Refuse any unsupported quantitative or model-generated statement.


def extractive_response07(run):
    if run['critic']['decision']!='evidence_only':
        return {'decision':run['critic']['decision'],
                'text':'No extractive evidence answer authorized for this task.',
                'citations':[]}
    cites=[]
    segments=[]
    for evidence_id in run['critic']['citation_ids']:
        assert evidence_id in corpus_by_id07
        r=corpus_by_id07[evidence_id]
        cites.append({'evidence_id':evidence_id,'doi':r['doi'],
                      'source_location':r['source_location']})
        segments.append(
            f"{r['claim_paraphrase']} [{evidence_id}] "
            f"DOI: {r['doi']}; source: {r['source_extent']}, {r['source_location']}. "
            f"Conditions: {r['conditions'] or 'not structured in reviewed claim'}."
        )
    return {'decision':'evidence_only','text':'\n'.join(segments),
            'citations':cites,'scope':'One publication; reported observation, not model prediction'}

answer07=extractive_response07(workflow_runs07[0])
assert len(answer07['citations'])==1
assert answer07['citations'][0]['evidence_id']=='E-P005-FE-C2H4-001'
assert all(c['evidence_id'] in workflow_runs07[0]['literature']['ids'] for c in answer07['citations'])
assert all(c['doi']==by_id07[c['evidence_id']]['doi'] for c in answer07['citations'])
print('EXTRACTIVE AGENT ANSWER WITH SOURCE')
print(answer07['text'])
print('Citation audit: PASSED | No LLM called')

EXTRACTIVE AGENT ANSWER WITH SOURCE
AgCu NW reached a maximum ethylene FE of 54.9% at -1.1 V vs RHE in a flow cell. [E-P005-FE-C2H4-001] DOI: 10.1038/s41467-024-54636-w; source: full_text, CO2RR performance; Fig. 2c; Supplementary Figs. 21b-23 and Table 5. Conditions: Flow cell; 1 M KOH; -1.1 V vs RHE; no iR correction.
Citation audit: PASSED | No LLM called


## 7.11 — Negative and reproducibility checks

**Why:** A successful demo is not enough. The tests deliberately include unsupported scientific requests, unknown candidate IDs, fake citations, and repeated identical requests. These are development tests—not independent model benchmarks or evidence of scientific superiority.

In [12]:
# PSEUDOCODE
# 1. Verify supported experimental requests return the correct unique citation.
# 2. Verify an unsupported durability or selectivity request abstains.
# 3. Verify an unregistered geometry never enters the proposal queue.
# 4. Verify a fabricated citation is rejected by the critic.
# 5. Verify identical inputs produce byte-for-byte identical serialized decisions.
# 6. Confirm bounded runs, no network, no energy, and no approval mutation.

smoke07=[]
def record07(case,ok): smoke07.append({'case':case,'passed':bool(ok)})

lit_ethylene07=run_workflow07(ResearchTask07('T07-01','source_question','What ethylene FE was reported for AgCu NW?'))
record07('01_source_linked_ethylene',lit_ethylene07['critic']['citation_ids']==('E-P005-FE-C2H4-001',))
lit_ethanol07=run_workflow07(ResearchTask07('T07-02','source_question','What ethanol FE was reported for Ag1Cu NW?'))
record07('02_source_linked_ethanol',lit_ethanol07['critic']['citation_ids']==('E-P005-FE-ETOH-001',))
ratio07=run_workflow07(ResearchTask07('T07-03','source_question','What ethanol-to-ethylene product ratios were reported?'))
record07('03_ratio_not_confused_with_FE',ratio07['critic']['citation_ids']==('E-DRAFT-001',))
durability07=run_workflow07(ResearchTask07('T07-04','source_question','What was the 100-hour durability of AgCu NW?'))
record07('04_unsupported_durability_abstains',durability07['critic']['decision']=='abstain')
rank07=run_workflow07(ResearchTask07('T07-05','predict_selectivity','Predict the best Cu-Ag catalyst for ethylene'))
record07('05_no_unsupported_selectivity_prediction',rank07['critic']['decision']=='abstain')
unknown07=run_workflow07(ResearchTask07('T07-06','structure_hypothesis','Review this hypothetical catalyst',('FAKE-AG-CANDIDATE',)))
record07('06_invalid_candidate_rejected',unknown07['critic']['decision']=='abstain')
try:
    preliminary_critic07(ResearchTask07('T07-07','source_question','What ethylene FE for AgCu NW?'),
                         EvidenceResult07(('FABRICATED-EVIDENCE-ID',),1,'pretend'),
                         DesignResult07((),'not_applicable'))
    fake_citation_rejected07=False
except AssertionError:
    fake_citation_rejected07=True
record07('07_fabricated_citation_rejected',fake_citation_rejected07)
repeated07=run_workflow07(workflow_tasks07[1])
record07('08_deterministic_reexecution',repeated07==workflow_runs07[1])
record07('09_step_budget_respected',all(len(r['trace'])<=3 for r in workflow_runs07))
record07('10_no_fake_scientific_approvals',all(
    not r['critic']['modeling_approved'] and not r['predicted_selectivity']
    and not r['external_api_called'] for r in workflow_runs07))

smoke_df07=pd.DataFrame(smoke07)
display(smoke_df07)
assert len(smoke_df07)==10 and smoke_df07['passed'].all(), 'Investigate a failing agent smoke test'
print('DEVELOPMENT ORCHESTRATION TESTS:',sum(smoke_df07['passed']),'/ 10 PASSED')
print('Not an independent critic benchmark, not a publication-level heldout test')

,case,passed
0,01_source_linked_ethylene,True
1,02_source_linked_ethanol,True
2,03_ratio_not_confused_with_FE,True
3,04_unsupported_durability_abstains,True
4,05_no_unsupported_selectivity_prediction,True
5,06_invalid_candidate_rejected,True
6,07_fabricated_citation_rejected,True
7,08_deterministic_reexecution,True
8,09_step_budget_respected,True
9,10_no_fake_scientific_approvals,True


DEVELOPMENT ORCHESTRATION TESTS: 10 / 10 PASSED
Not an independent critic benchmark, not a publication-level heldout test


## 7.12 — Agent execution isolation and optional future LLM adapters

**Why:** These roles are deterministic functions. A later LLM adapter may generate *draft hypotheses* but must never bypass tool permissions, citation checks, cost budgets, or human approval. No API calls, model downloads, autonomous loops, or scraping occur here.

In [13]:
# PSEUDOCODE
# 1. Inspect hard limits and explicitly disabled capabilities.
# 2. Confirm a future model adapter would require a separate permission gate.
# 3. Confirm human review and experimental modeling remain outside agent tools.
# 4. Do not execute any model, network call or write operation.

adapter_contract07={
    'execution_mode':'offline_deterministic_roles',
    'generative_llm_integrated':False,
    'network_enabled':ALLOW_EXTERNAL_LLM07,
    'atomistic_energy_or_force_calculator_run':RUN_COMPUTATIONAL_SCORING07,
    'maximum_agent_tool_steps':MAX_AGENT_STEPS07,
    'maximum_structure_candidates':MAX_CANDIDATES07,
    'maximum_retrieved_records':MAX_EVIDENCE07,
    'human_evidence_approval_mutable':False,
    'structure_modeling_approval_mutable':False,
    'untrusted_retrieval_text_is_data_not_instructions':True,
    'future_llm_drafts_require_independent_scientific_review':True,
}
assert adapter_contract07['network_enabled'] is False
assert adapter_contract07['atomistic_energy_or_force_calculator_run'] is False
assert adapter_contract07['human_evidence_approval_mutable'] is False
print('AGENT CAPABILITY / PERMISSION AUDIT')
print(json.dumps(adapter_contract07,indent=2))

AGENT CAPABILITY / PERMISSION AUDIT
{
  "execution_mode": "offline_deterministic_roles",
  "generative_llm_integrated": false,
  "network_enabled": false,
  "atomistic_energy_or_force_calculator_run": false,
  "maximum_agent_tool_steps": 3,
  "maximum_structure_candidates": 2,
  "maximum_retrieved_records": 3,
  "human_evidence_approval_mutable": false,
  "structure_modeling_approval_mutable": false,
  "untrusted_retrieval_text_is_data_not_instructions": true,
  "future_llm_drafts_require_independent_scientific_review": true
}


## 7.13 — Optional isolated export of the agent decision and tool ledgers

**Default:** `EXPORT_AGENT_TRACES07 = False`. Running the notebook again is read-only. If you manually opt in, it publishes a new folder `data/processed/07_multi_agent_prototype/` with a machine-readable agent ledger and a candidate progression queue, **refusing overwrites**. The folder includes *no new scientific approvals* or trained model predictions.

**Before enabling:** Inspect the three case outputs, ten smoke tests, provenance checks, and the contents of every trace. Keep the switch `False` in Git after the one-time export.

In [ ]:
# PSEUDOCODE
# 1. Default all local file writes to OFF.
# 2. If enabled manually, refuse to overwrite any prior agent export.
# 3. Serialize only deterministic traces and proposal states to a temporary folder.
# 4. Verify record counts, source hashes, and conservative scientific flags.
# 5. Publish the new isolated folder atomically; never touch prior notebook exports.

EXPORT_AGENT_TRACES07 = False
AGENT_EXPORT_ROOT07 = ROOT / 'data/processed/07_multi_agent_prototype'

if not EXPORT_AGENT_TRACES07:
    print('MULTI-AGENT TRACE EXPORT: DISABLED — no files written')
    print('Optional destination:',AGENT_EXPORT_ROOT07)
else:
    import tempfile
    import shutil
    assert not AGENT_EXPORT_ROOT07.exists(), 'Export folder already exists; overwrite refused'
    AGENT_EXPORT_ROOT07.parent.mkdir(parents=True,exist_ok=True)
    staging_parent07=Path(tempfile.mkdtemp(
        prefix='.07_agent_staging_',dir=str(AGENT_EXPORT_ROOT07.parent)))
    try:
        staged07=staging_parent07 / AGENT_EXPORT_ROOT07.name
        staged07.mkdir()
        lines07=[canonical07(r) for r in workflow_runs07]
        payload07=('\n'.join(lines07)+'\n').encode('utf-8')
        (staged07/'07_agent_decision_ledger.jsonl').write_bytes(payload07)
        progress_df07.to_csv(staged07/'07_candidate_review_queue.csv',index=False)
        manifest07={
            'campaign_id':'CU-001',
            'notebook':'07_Multi_Agent_Workflow',
            'run_count':len(workflow_runs07),
            'tool_steps_per_run':MAX_AGENT_STEPS07,
            'source_corpus_sha256':corpus_manifest07['corpus_sha256'],
            'agent_ledger_sha256':hashlib.sha256(payload07).hexdigest(),
            'independent_critic_benchmark_completed':False,
            'production_rag_validated':False,
            'human_approvals_modified':False,
            'predicted_catalytic_selectivity':False,
        }
        (staged07/'07_agent_manifest.json').write_text(json.dumps(manifest07,indent=2)+'\n',encoding='utf-8')
        check07=[json.loads(x) for x in (staged07/'07_agent_decision_ledger.jsonl').read_text('utf-8').splitlines()]
        assert len(check07)==len(workflow_runs07)==3
        assert all(len(x['trace'])==3 for x in check07)
        assert not any(x['critic']['modeling_approved'] or x['predicted_selectivity'] for x in check07)
        staged07.rename(AGENT_EXPORT_ROOT07)
    finally:
        shutil.rmtree(staging_parent07,ignore_errors=True)
    print('AGENT TRACE EXPORT COMPLETE:',AGENT_EXPORT_ROOT07)
    print('Source evidence, atomic geometry and human-review ledgers not modified')

AGENT TRACE EXPORT COMPLETE: /Users/nsolebna/202609_Ironhack/Week 8/Project 3/CuSelect-AI/data/processed/07_multi_agent_prototype
Source evidence, atomic geometry and human-review ledgers not modified


## 7.14 — Scientific findings, limitations, and handoff to Notebook 08

### Demonstrated
- Loaded versioned Notebook 02, 04, 05, 06 handoffs and revalidated their integrity.
- Coordinated deterministic **Literature**, **Design**, and **Scientific Critic** roles under explicit tool allowlists and a three-step budget.
- Returned citation-linked experimental evidence without claiming that a publication establishes general catalyst superiority.
- Proposed two previously registered, matched-composition Cu(100) Ag-pair structures as **unrelaxed hypotheses**, not performance-ranked candidates.
- Preserved a typed, replayable execution trace, candidate progression, and auditable abstentions.

### Not demonstrated
- Independent LLM agents, autonomous scientific reasoning, or access to external resources.
- Relaxation, energy, adsorption or selectivity calculations for the eight geometries.
- An independently validated scientific Critic, improvement over a baseline, or a leakage-safe benchmark.
- A multi-paper or broad literature corpus; retrieval coverage still comes from one publication.

### Notebook 08 — Scientific Critic and Active Learning

The next stage should implement independent objections, evidence-specific falsification tests, bounded request-for-evidence actions, and explicit **false advancement vs. false rejection** metrics. It must not simply rubber-stamp these deterministic role outputs. Notebook 09 will measure actual comparative performance, uncertainty, cost, and publication-level holdout when data suffices.

**Completion gate for this notebook:** Fresh-kernel Run All passes, outputs and original records are unchanged, all ten development tests pass, and optional trace export is reviewed before use. Retain `EXPORT_AGENT_TRACES07 = False` in the committed notebook.

In [17]:
# PSEUDOCODE
# 1. Summarize what the deterministic agent prototype has actually demonstrated.
# 2. Verify all ten development smoke checks and three sample workflow decisions.
# 3. Keep production, ML, RAG, and critic-readiness claims false.
# 4. Report any existing optional ledger export without creating or modifying files.
# 5. Name the original next notebook and pending scientific validations.

summary07={
    'campaign_id':'CU-001',
    'notebook':'07_Multi_Agent_Workflow',
    'verified_experimental_records':len(corpus07),
    'source_publications':len({r['paper_id'] for r in corpus07}),
    'registered_unrelaxed_geometries':len(registry07),
    'agent_roles':list(AGENT_TOOLS07),
    'workflow_runs':len(workflow_runs07),
    'max_steps_per_run':MAX_AGENT_STEPS07,
    'run_decisions':[r['critic']['decision'] for r in workflow_runs07],
    'development_smoke_tests_passed':int(smoke_df07['passed'].sum()),
    'deterministic_replay_passed':repeated07==workflow_runs07[1],
    'human_approval_modified':False,
    'atomistic_energies_computed':False,
    'catalytic_selectivity_predicted':False,
    'modeling_approved':False,
    'llm_agents_executed':False,
    'independent_critic_benchmark_completed':False,
    'production_agent_system_validated':False,
    'optional_trace_export_exists':AGENT_EXPORT_ROOT07.exists(),
    'next_notebook':'08_Scientific_Critic_and_Active_Learning',
}
assert summary07['development_smoke_tests_passed']==10
assert summary07['run_decisions']==['evidence_only','hypothesis_only','abstain']
assert summary07['production_agent_system_validated'] is False
print('NOTEBOOK 07 — MULTI-AGENT WORKFLOW SUMMARY')
print(json.dumps(summary07,indent=2))
print('Reproducible coordination prototype complete; scientific critic benchmark remains pending')

NOTEBOOK 07 — MULTI-AGENT WORKFLOW SUMMARY
{
  "campaign_id": "CU-001",
  "notebook": "07_Multi_Agent_Workflow",
  "verified_experimental_records": 3,
  "source_publications": 1,
  "registered_unrelaxed_geometries": 8,
  "agent_roles": [
    "literature_agent",
    "design_agent",
    "scientific_critic"
  ],
  "workflow_runs": 3,
  "max_steps_per_run": 3,
  "run_decisions": [
    "evidence_only",
    "hypothesis_only",
    "abstain"
  ],
  "development_smoke_tests_passed": 10,
  "deterministic_replay_passed": true,
  "human_approval_modified": false,
  "atomistic_energies_computed": false,
  "catalytic_selectivity_predicted": false,
  "modeling_approved": false,
  "llm_agents_executed": false,
  "independent_critic_benchmark_completed": false,
  "production_agent_system_validated": false,
  "optional_trace_export_exists": true,
  "next_notebook": "08_Scientific_Critic_and_Active_Learning"
}
Reproducible coordination prototype complete; scientific critic benchmark remains pending
